In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE servers (
    server_id TEXT PRIMARY KEY,
    region TEXT,
    tier TEXT
);
INSERT INTO servers VALUES
    ('srv-01','us-east','gold'),
    ('srv-02','us-east','silver'),
    ('srv-03','us-west','gold'),
    ('srv-04','us-west','bronze'),
    ('srv-05','us-east','silver'),
    ('srv-06','eu-west','gold');

CREATE TABLE daily_metrics (
    server_id TEXT,
    report_date TEXT,
    avg_cpu REAL
);
INSERT INTO daily_metrics VALUES
    ('srv-01','2026-02-26',72.5),
    ('srv-01','2026-02-27',74.1),
    ('srv-02','2026-02-26',45.2),
    ('srv-02','2026-02-27',48.3),
    ('srv-03','2026-02-26',91.3),
    ('srv-03','2026-02-27',89.7),
    ('srv-04','2026-02-26',33.1),
    ('srv-05','2026-02-26',55.0);
    -- srv-06 has NO metrics (gap detection test)
    -- srv-04 and srv-05 missing 2026-02-27 (partial gap)
""")
print("Sample database ready!")
print("Servers: 6 rows | daily_metrics: 8 rows")
print()
print(pd.read_sql_query("SELECT * FROM servers", conn).to_string(index=False))

# SQL — Complex JOINs
**Day 3 — SQL Module**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>Core Insight:</strong> Most candidates know INNER JOIN. What separates senior engineers:
self-joins, cross joins for combinations, and anti-joins for gap detection.
These show up constantly in data quality, missing-data detection, and combinatorial analysis.
</div>

**Run j00-setup first** to create the sample database.

## Self Join — Compare Rows Within the Same Table

A self join joins a table to itself. Use it to compare rows within the same table:
- Year-over-year comparison
- Server pairs in the same region
- Sequential record analysis (day-over-day delta)

In [ ]:
# Self Join: find server pairs in the same region with CPU gap > 20%

sql_self = """
SELECT
    a.server_id AS server_1,
    b.server_id AS server_2,
    a.region,
    ROUND(a.avg_cpu, 1) AS cpu_1,
    ROUND(b.avg_cpu, 1) AS cpu_2,
    ROUND(ABS(a.avg_cpu - b.avg_cpu), 1) AS cpu_gap
FROM daily_metrics a
JOIN daily_metrics b
    ON a.report_date = b.report_date
    AND a.server_id < b.server_id       -- avoid (A,B) and (B,A) duplicates
JOIN servers sa ON a.server_id = sa.server_id
JOIN servers sb ON b.server_id = sb.server_id
WHERE sa.region = sb.region             -- same region
  AND a.report_date = '2026-02-26'
  AND ABS(a.avg_cpu - b.avg_cpu) > 20
ORDER BY cpu_gap DESC
"""

result = pd.read_sql_query(sql_self, conn)
print("Server pairs in same region with CPU gap > 20% (date: 2026-02-26):")
print(result.to_string(index=False))
print()
print("Note: 'a.server_id < b.server_id' prevents (srv-01,srv-02) and (srv-02,srv-01) both appearing.")

# Day-over-day delta using self join
sql_dod = """
SELECT
    a.server_id,
    a.report_date AS today,
    b.report_date AS yesterday,
    ROUND(a.avg_cpu, 1) AS today_cpu,
    ROUND(b.avg_cpu, 1) AS yesterday_cpu,
    ROUND(a.avg_cpu - b.avg_cpu, 1) AS delta
FROM daily_metrics a
JOIN daily_metrics b
    ON a.server_id = b.server_id
    AND b.report_date = date(a.report_date, '-1 day')
ORDER BY ABS(a.avg_cpu - b.avg_cpu) DESC
"""

print()
print("Day-over-day CPU delta (self join on server_id + date offset):")
print(pd.read_sql_query(sql_dod, conn).to_string(index=False))

## Anti-Join — Find Missing Records (Data Gap Detection)

Anti-join returns rows from the left table with NO matching row in the right table.
This is the core pattern for:
- Servers that reported no data yesterday
- Products with no sales
- Users who never logged in

**Three equivalent methods — choose based on your database:**
1. `LEFT JOIN + WHERE right.key IS NULL` — most portable
2. `NOT EXISTS (subquery)` — often most readable
3. `NOT IN (subquery)` — **dangerous with NULLs** — avoid unless you know subquery has no NULLs

In [ ]:
# Anti-Join: servers that reported NO data on 2026-02-27

# Method 1: LEFT JOIN + IS NULL (most portable, works everywhere)
sql_antijoin1 = """
SELECT s.server_id, s.region, s.tier
FROM servers s
LEFT JOIN daily_metrics m
    ON s.server_id = m.server_id
    AND m.report_date = '2026-02-27'
WHERE m.server_id IS NULL
ORDER BY s.server_id
"""
result1 = pd.read_sql_query(sql_antijoin1, conn)
print("Anti-join (LEFT JOIN + IS NULL): servers missing 2026-02-27 data:")
print(result1.to_string(index=False))

# Method 2: NOT EXISTS (readable, safe with NULLs)
sql_antijoin2 = """
SELECT server_id, region, tier
FROM servers s
WHERE NOT EXISTS (
    SELECT 1 FROM daily_metrics m
    WHERE m.server_id = s.server_id
    AND m.report_date = '2026-02-27'
)
ORDER BY server_id
"""
result2 = pd.read_sql_query(sql_antijoin2, conn)
print()
print("Anti-join (NOT EXISTS): same result, different approach:")
print(result2.to_string(index=False))
print()
print("Both return the same servers: those with no row in daily_metrics for 2026-02-27.")
print("srv-06 has NO data at all. srv-04, srv-05 only have 2026-02-26 data.")

## Cross Join — Generate All Combinations

Cross join produces every combination of rows from two tables.
Use cases:
- Generate a complete grid (server × metric_type)
- Create date × server combinations for gap detection
- Combinatorial analysis

```sql
-- Generate all (server, metric) combinations
SELECT s.server_id, m.metric_type
FROM servers s
CROSS JOIN (VALUES ('cpu'), ('memory'), ('disk')) AS m(metric_type);
-- Then LEFT JOIN actual metrics to find what's missing
```

In [ ]:
# Cross Join: generate all (server, date) combinations, then find gaps

# All servers × all dates in our range
sql_cross = """
SELECT s.server_id, d.report_date
FROM servers s
CROSS JOIN (
    SELECT DISTINCT report_date FROM daily_metrics
) d
ORDER BY s.server_id, d.report_date
"""
all_combos = pd.read_sql_query(sql_cross, conn)
print(f"Cross join: {len(all_combos)} (server, date) combinations")
print(all_combos.head(8).to_string(index=False))

# Now find gaps: combinations that exist in cross join but NOT in daily_metrics
sql_gaps = """
SELECT s.server_id, d.report_date, s.tier
FROM servers s
CROSS JOIN (
    SELECT DISTINCT report_date FROM daily_metrics
) d
WHERE NOT EXISTS (
    SELECT 1 FROM daily_metrics m
    WHERE m.server_id = s.server_id AND m.report_date = d.report_date
)
ORDER BY d.report_date, s.server_id
"""
gaps = pd.read_sql_query(sql_gaps, conn)
print()
print(f"Missing data gaps ({len(gaps)} server-date combinations):")
print(gaps.to_string(index=False))

## Interview Q&A

**Q: Why is `NOT IN` dangerous with NULLs?**
A: If the subquery returns any NULL value, the entire NOT IN comparison returns NULL for every row — effectively returning no results. Always use NOT EXISTS or LEFT JOIN + IS NULL instead. Rule: never use NOT IN with a subquery unless you can 100% guarantee no NULLs in the subquery result.

**Q: Self-join use cases in data engineering?**
A: Year-over-year comparison (join table to itself on year offset), sequential record pairing (day N vs day N-1), hierarchy traversal (employee to their manager in the same table), finding duplicate records.

**Q: What's a hash join?**
A: Build a hash table from the smaller table, probe it with the larger table. O(n+m). Most databases choose this automatically for large joins. Contrast with nested loop join: O(n×m) — only used for small tables or indexed lookups.

**Q: USING vs ON — when do you use each?**
A: `USING (server_id)` when the join column has the same name in both tables — cleaner, only appears once in SELECT *. `ON a.id = b.server_id` when column names differ or when you want explicitness. Prefer ON for clarity in complex queries.

**Q: What is a broadcast join in Spark?**
A: A small table is replicated to all Spark executors — avoids the expensive shuffle that a regular join would require. Use when one table is small (< a few hundred MB). `F.broadcast(small_df)` hint in PySpark.

## The Citi Angle

**Data gap detection was a daily operation.** At Citi, 6,000 servers reporting telemetry — any gap meant an agent went down, a network issue occurred, or a server was decommissioned without notice.

**The morning data quality check:**
```sql
-- Run every morning: find servers that didn't report yesterday
SELECT s.server_id, s.region, s.tier,
       MAX(m.report_date) AS last_seen
FROM servers s
LEFT JOIN daily_metrics m ON s.server_id = m.server_id
GROUP BY s.server_id, s.region, s.tier
HAVING MAX(m.report_date) < date('now', '-1 day')
    OR MAX(m.report_date) IS NULL
ORDER BY last_seen NULLS FIRST
```

This query combines LEFT JOIN (for servers with no data at all) with HAVING (for servers whose last report is stale). The result was fed into an alerting pipeline.

**Interview line:** *"Anti-join was the foundation of our data quality monitoring at Citi — LEFT JOIN + IS NULL pattern to detect the 'silent failure' of a server that stopped reporting. We reduced gap detection time from hours to minutes by running this query on a schedule."*